# RAG - Regulamentações de IA em Universidades Federais


## 1. Instalação de dependências

In [ ]:
%%capture
!pip install -q \
    pymupdf \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-text-splitters \
    langchain-chroma \
    chromadb \
    rank_bm25 \
    sentence-transformers \
    einops \
    groq

print("Dependencias instaladas!")

## 2. Configuração

In [ ]:
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    print("Groq API key carregada dos Secrets")
except Exception:
    GROQ_API_KEY = input("Cole sua Groq API key: ").strip()

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HuggingFace token carregado dos Secrets")
except Exception:
    HF_TOKEN = input("Cole seu HuggingFace token: ").strip()

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

LLM_MODEL       = "llama-3.3-70b-versatile"
EMBEDDING_MODEL = "paraphrase-multilingual-mpnet-base-v2"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 400

TOP_K_EACH = 10

# K final varia por tipo de pergunta
TOP_K_ESPECIFICA  = 5    # 1 universidade mencionada
TOP_K_COMPARATIVA = 8    # 2+ universidades mencionadas
TOP_K_GERAL       = 12   # nenhuma universidade mencionada

print(f"  LLM           : {LLM_MODEL}")
print(f"  Embeddings    : {EMBEDDING_MODEL}")
print(f"  Reranker      : {RERANKER_MODEL}")
print(f"  K especifica  : {TOP_K_ESPECIFICA}")
print(f"  K comparativa : {TOP_K_COMPARATIVA}")
print(f"  K geral       : {TOP_K_GERAL}")


## 3. Upload e Extração dos PDFs

In [ ]:
import fitz
from google.colab import files
import os

os.makedirs("/content/pdfs", exist_ok=True)

print("Selecione os PDFs para upload...")
uploaded = files.upload()

for filename in uploaded.keys():
    with open(f"/content/pdfs/{filename}", "wb") as f:
        f.write(uploaded[filename])

print(f"\n{len(uploaded)} arquivo(s) carregado(s):")
for name in uploaded.keys():
    print(f"   * {name}")

In [ ]:
import re

# Mapa de aliases -> nome canonico
# Substrings perigosas como "para", "acre", "parana" foram removidas
ALIASES_UNIVERSIDADE = {
    "Unifesp": "UNIFESP", "unifesp":"UNIFESP", "sao paulo": "UNIFESP", "são paulo": "UNIFESP",
    "ufpb": "UFPB", "paraiba": "UFPB", "paraíba": "UFPB",
    "ufc": "UFC", "ceara": "UFC", "ceára": "UFC",
    "ufba": "UFBA", "bahia": "UFBA",
    "ufdpar": "UFDPAR", "delta do parnaiba": "UFDPAR",
    "uff": "UFF", "fluminense": "UFF",
    "ufg": "UFG", "goias": "UFG", "goiás": "UFG",
    "ufma": "UFMA", "maranhao": "UFMA", "maranhão": "UFMA",
    "ufmg": "UFMG", "minas gerais": "UFMG",
    "ufms": "UFMS",
    "ufop": "UFOP", "ouro preto": "UFOP",
    "ufrj": "UFRJ", "rio de janeiro": "UFRJ",
    "ufu": "UFU", "uberlandia": "UFU", "uberlândia": "UFU",
    "unifal": "UNIFAL MG", "alfenas": "UNIFAL MG",
    "unir": "UNIR", "rondonia": "UNIR", "rondônia": "UNIR",
    "uffrj": "UFFRJ"
}

def inferir_universidade(nome_arquivo: str) -> str:
    """Identifica a universidade pelo nome do arquivo."""
    nome_norm = nome_arquivo.lower().replace("-", "_").replace(" ", "_")
    for alias in sorted(ALIASES_UNIVERSIDADE, key=len, reverse=True):
        pattern = r"\b" + re.escape(alias.replace(" ", "_")) + r"\b"
        if re.search(pattern, nome_norm):
            return ALIASES_UNIVERSIDADE[alias]
    return os.path.splitext(nome_arquivo)[0].replace("_", " ").replace("-", " ")

def extrair_universidades_da_query(query: str) -> list[str]:
    """Detecta TODAS as universidades mencionadas na query."""
    query_norm = query.lower().strip()
    encontradas = []
    for alias in sorted(ALIASES_UNIVERSIDADE, key=len, reverse=True):
        pattern = r"\b" + re.escape(alias) + r"\b"
        if re.search(pattern, query_norm):
            nome = ALIASES_UNIVERSIDADE[alias]
            if nome not in encontradas:
                encontradas.append(nome)
    return encontradas

def classificar_query(universidades: list[str]) -> tuple[str, int]:
    """
    Classifica a query e retorna (tipo, top_k_final).
      especifica  -> 1 universidade  -> TOP_K_ESPECIFICA
      comparativa -> 2+              -> TOP_K_COMPARATIVA
      geral       -> nenhuma         -> TOP_K_GERAL
    """
    if len(universidades) >= 2:
        return "comparativa", TOP_K_COMPARATIVA
    if len(universidades) == 1:
        return "especifica", TOP_K_ESPECIFICA
    return "geral", TOP_K_GERAL

def extrair_texto_pdf(caminho_pdf: str) -> list:
    """Extrai texto de um PDF preservando metadados por pagina."""
    doc = fitz.open(caminho_pdf)
    nome_arquivo = os.path.basename(caminho_pdf)
    universidade = inferir_universidade(nome_arquivo)
    paginas = []
    for num_pagina in range(len(doc)):
        texto = doc[num_pagina].get_text().strip()
        if len(texto) > 50:
            paginas.append({
                "texto":        texto,
                "pagina":       num_pagina + 1,
                "arquivo":      nome_arquivo,
                "universidade": universidade
            })
    doc.close()
    return paginas


todas_paginas = []
pdf_dir = "/content/pdfs"

for arquivo in sorted(os.listdir(pdf_dir)):
    if arquivo.endswith(".pdf"):
        paginas = extrair_texto_pdf(os.path.join(pdf_dir, arquivo))
        todas_paginas.extend(paginas)
        uni = paginas[0]["universidade"] if paginas else "?"
        print(f"  {arquivo}  ->  [{uni}]  ({len(paginas)} pag.)")

print(f"\nTotal: {len(todas_paginas)} paginas extraidas")
print("Dica: nome errado? Adicione o alias em ALIASES_UNIVERSIDADE acima.")

## 4. Chunking com prefixo semântico


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

documentos = []
for pagina in todas_paginas:
    texto_com_prefixo = f"Regulamentacao de IA da {pagina['universidade']}: {pagina['texto']}"
    documentos.append(Document(
        page_content=texto_com_prefixo,
        metadata={
            "arquivo":      pagina["arquivo"],
            "universidade": pagina["universidade"],
            "pagina":       pagina["pagina"]
        }
    ))

chunks = splitter.split_documents(documentos)

print(f"Chunking concluido:")
print(f"   Paginas : {len(documentos)}")
print(f"   Chunks  : {len(chunks)}")
print(f"\n   Exemplo : {chunks[0].page_content[:200]}...")

## 5. Indexação: BM25 + ChromaDB


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

print(f"Carregando embeddings: {EMBEDDING_MODEL}")
try:
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True}
    )
    print("GPU")
except Exception:
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    print("CPU (GPU nao disponivel)")

In [ ]:
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
import chromadb

chroma_client = chromadb.PersistentClient(path="/content/chroma_db")

print("Criando indice ChromaDB...")
chroma_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    client=chroma_client,
    collection_name="regulamentacoes_ia"
)
print(f"ChromaDB: {chroma_store._collection.count()} vetores indexados")

print("\nCriando indice BM25...")
bm25_retriever   = BM25Retriever.from_documents(chunks)
bm25_retriever.k = TOP_K_EACH
print(f"BM25: {len(chunks)} documentos indexados")

## 6. Retrieval Híbrido com Fusão de N Retrievers

In [ ]:
from typing import List

def rrf_multi(listas: list, k: int = 60) -> List[Document]:
    """
    RRF generico para N listas.
    score = soma(1 / (k + rank)) para cada lista em que o chunk aparece.
    """
    scores  = {}
    doc_map = {}

    for lista in listas:
        for rank, doc in enumerate(lista):
            chave = doc.page_content[:120]
            scores[chave]  = scores.get(chave, 0) + 1 / (k + rank + 1)
            doc_map[chave] = doc

    ordenados = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[chave] for chave, _ in ordenados]


def chroma_filtrado(query: str, universidade: str, k: int) -> List[Document]:
    """Busca semantica no ChromaDB filtrada por universidade."""
    try:
        return chroma_store.similarity_search(
            query, k=k, filter={"universidade": universidade}
        )
    except Exception:
        return []


def retrieval_hibrido(query: str, verbose: bool = False) -> List[Document]:
    """
    Fusao de N retrievers com RRF e k dinamico por tipo de query.
    Retorna os candidatos para o reranker (sem cortar ainda).
    """
    universidades        = extrair_universidades_da_query(query)
    tipo, top_k_dinamico = classificar_query(universidades)

    if verbose:
        print(f"  [query] tipo: {tipo} | universidades: {universidades or ['nenhuma']} | k final: {top_k_dinamico}")

    res_bm25   = bm25_retriever.invoke(query)
    res_global = chroma_store.similarity_search(query, k=TOP_K_EACH)

    res_filtrados = []
    for uni in universidades:
        res = chroma_filtrado(query, uni, TOP_K_EACH)
        if res:
            res_filtrados.append(res)
            if verbose:
                print(f"  [chroma filtrado] {uni}: {len(res)} chunks")
        else:
            res_filtrados.append(res_global)
            if verbose:
                print(f"  [chroma filtrado] {uni}: vazio -> fallback global")

    if not res_filtrados:
        res_filtrados = [res_global]

    candidatos = rrf_multi([res_bm25, res_global] + res_filtrados)

    if verbose:
        print(f"  [rrf] {len(candidatos)} candidatos unicos")

    return candidatos

## 7. Reranker

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    RERANKER_MODEL,
    max_length=512,
)

In [ ]:
def rerankar(query: str, candidatos: List[Document], top_k: int) -> List[Document]:
    """
    Reordena candidatos com Jina Reranker v2.
    Retorna score unico por par, sem necessidade de indice [1].
    """
    if not candidatos:
        return []
    pares     = [(query, doc.page_content) for doc in candidatos]
    scores    = reranker.predict(pares)
    ordenados = sorted(zip(scores, candidatos), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ordenados[:top_k]]


print("\nTeste rapido do reranker...")
_q     = "uso de inteligencia artificial em trabalhos academicos"
_cands = retrieval_hibrido(_q)
_uni   = extrair_universidades_da_query(_q)
_, _k  = classificar_query(_uni)
_final = rerankar(_q, _cands, _k)
print(f"Reranker OK -> {len(_cands)} candidatos -> {len(_final)} chunks finais")

## 8. Geração de Resposta com LLM (Groq)

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatGroq(
    model=LLM_MODEL,
    temperature=0.1,
    groq_api_key=GROQ_API_KEY
)

SYSTEM_PROMPT = """Voce e um assistente especializado em regulamentacoes sobre uso de Inteligencia Artificial em universidades federais brasileiras.

Responda com base EXCLUSIVAMENTE nos trechos de documentos fornecidos.
Seja claro, objetivo e sempre cite de qual universidade/documento veio a informacao.
Para perguntas comparativas, organize a resposta por universidade.
Se a informacao nao estiver nos trechos, diga explicitamente que nao encontrou nos documentos.
Responda sempre em portugues brasileiro."""


def formatar_contexto(chunks: List[Document]) -> str:
    trechos = []
    for i, chunk in enumerate(chunks):
        trechos.append(
            f"[Trecho {i+1}] {chunk.metadata['universidade']} "
            f"(Pag. {chunk.metadata['pagina']}):\n{chunk.page_content}"
        )
    return "\n\n".join(trechos)


def responder(pergunta: str, verbose: bool = False) -> dict:
    """
    Pipeline completo:
    fusao de N retrievers -> RRF com k dinamico -> Jina Reranker v2 -> LLM
    """
    universidades        = extrair_universidades_da_query(pergunta)
    tipo, top_k_dinamico = classificar_query(universidades)

    candidatos        = retrieval_hibrido(pergunta, verbose=verbose)
    chunks_relevantes = rerankar(pergunta, candidatos, top_k_dinamico)
    contexto          = formatar_contexto(chunks_relevantes)

    user_prompt = (
        "Com base nos seguintes trechos de regulamentacoes universitarias sobre IA:\n\n"
        + contexto
        + f"\n\nPergunta: {pergunta}"
    )

    resposta = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ])

    fontes = sorted(set(
        f"{c.metadata['universidade']} (pag. {c.metadata['pagina']})"
        for c in chunks_relevantes
    ))

    if verbose:
        print(f"  Tipo de query     : {tipo}")
        print(f"  K dinamico        : {top_k_dinamico}")
        print(f"  Chunks no contexto: {len(chunks_relevantes)}")
        print(f"  Fontes            : {fontes}")

    return {
        "pergunta": pergunta,
        "tipo":     tipo,
        "top_k":    top_k_dinamico,
        "resposta": resposta.content,
        "fontes":   fontes,
        "chunks":   chunks_relevantes,
        "contexto": contexto
    }

## 9. Avaliação do Retriever

Avaliação científica do pipeline com métricas clássicas de recuperação de informação:

| Métrica | O que mede |
|---|---|
| **Precision@K** | Dos K chunks recuperados, quantos eram relevantes? |
| **Recall@K** | Dos chunks relevantes existentes, quantos foram recuperados? |
| **MRR** | O primeiro chunk relevante aparece em que posição? |


In [ ]:
GABARITO = {
    "a ufpb permite o uso de ia para formatar textos?": {
        "relevantes": [
            "UFPB_p2_af720d"
        ],
        "tipo": "especifica"
    },
    "qual é a comissão responsável por analisar denúncias de plágio no âmbito dos centros de ensino da ufpb?": {
        "relevantes": [
            "UFPB_p3_8f7f48",
            "UFPB_p3_071d4d",
            "UFPB_p3_f02b81"
        ],
        "tipo": "especifica"
    },
    "qual o percentual de similaridade em um trabalho que é considerado aceitável e passível de aprovação na ufpb?": {
        "relevantes": [
            "UFPB_p4_295251",
            "UFPB_p4_f84cc9"
        ],
        "tipo": "especifica"
    },
    "quais são as possíveis sanções aplicáveis em caso de procedência de denúncia de plágio na ufpb?": {
        "relevantes": [
            "UFPB_p6_bc4fad",
            "UFPB_p6_1e7789"
        ],
        "tipo": "especifica"
    },
    "quais os usos de ia vedados na ufpb?": {
        "relevantes": [
            "UFPB_p2_8c8f71",
            "UFPB_p3_8f7f48"
        ],
        "tipo": "especifica"
    },
    "qual a ferramenta de detecção de similaridade recomendada pela ufc?": {
        "relevantes": [
            "UFC_p3_bcf4ba"
        ],
        "tipo": "especifica"
    },
    "quais são os documentos obrigatórios que devem ser anexados ao pedido de defesa na ufc?": {
        "relevantes": [
            "UFC_p3_bcf4ba",
            "UFC_p1_c0e441"
        ],
        "tipo": "especifica"
    },
    "pode utilizar ia para redigir conclusões na ufc?": {
        "relevantes": [
            "UFC_p2_d8acab",
            "UFC_p4_2e29ae"
        ],
        "tipo": "especifica"
    },
    "na ufc, o que ocorre se o docente não explicitar no plano de ensino se o uso de ia é permitido ou não?": {
        "relevantes": [
            "UFC_p2_da335f"
        ],
        "tipo": "especifica"
    },
    "quais os usos de ia vedados na ufc?": {
        "relevantes": [
            "UFC_p2_5e60a8",
            "UFC_p2_d8acab",
            "UFC_p4_2e29ae"
        ],
        "tipo": "especifica"
    },
    "a declaração de uso da ia é obrigatoria na ufms?": {
        "relevantes": [
            "UFMS_p4_a6cbfc"
        ],
        "tipo": "especifica"
    },
    "ferramentas de inteligência artificial podem ser listadas como autores em trabalhos acadêmicos na ufms?": {
        "relevantes": [
            "UFMS_p6_0b03ec"
        ],
        "tipo": "especifica"
    },
    "quais são os órgãos responsáveis pela governança institucional de ia na ufms?": {
        "relevantes": [
            "UFMS_p4_91de85",
            "UFMS_p3_5eb211"
        ],
        "tipo": "especifica"
    },
    "como o estudante deve agir caso não haja instrução formal do professor sobre o uso de ia em uma atividade na ufms?": {
        "relevantes": [
            "UFMS_p4_a6cbfc"
        ],
        "tipo": "especifica"
    },
    "como a ia pode ser usada nas atividades administrativas da ufms?": {
        "relevantes": [
            "UFMS_p6_0b03ec",
            "UFMS_p6_fd04f1"
        ],
        "tipo": "especifica"
    },
    "qual o nome da comissão responsável por assessorar o reitor da ufrj na elaboração de diretrizes sobre inteligência artificial?": {
        "relevantes": [
            "UFRJ_p2_4ea2c3"
        ],
        "tipo": "especifica"
    },
    "de acordo com a ufrj, o estudante pode usar ia se o professor for omisso sobre o assunto?": {
        "relevantes": [
            "UFRJ_p12_1d5cd6"
        ],
        "tipo": "especifica"
    },
    "onde o docente da ufrj deve definir o que constitui uso aceitável ou inaceitável de ferramentas de ia em sua disciplina?": {
        "relevantes": [
            "UFRJ_p5_98f2ca"
        ],
        "tipo": "especifica"
    },
    "em qual parte de um artigo ou trabalho acadêmico o pesquisador da ufrj deve informar o uso de ferramentas de ia?": {
        "relevantes": [
            "UFRJ_p5_5cedd4"
        ],
        "tipo": "especifica"
    },
    "quais os principios gerais do uso de ia na ufrj?": {
        "relevantes": [
            "UFRJ_p5_978558",
            "UFRJ_p5_ea7888"
        ],
        "tipo": "especifica"
    },
    "a ufdpar preve declaração de uso de ia?": {
        "relevantes": [
            "UFDPAR_p8_8157ae"
        ],
        "tipo": "especifica"
    },
    "A quem se aplica a Política de Inteligência Artificial estabelecida pela UFDPar?": {
        "relevantes": [
            "UFDPAR_p3_e4296b",
            "UFDPAR_p11_dae0ea",
        ],
        "tipo": "especifica"
    },
    "quais tipos de ferramentas de ia são consideradas de \"baixo risco\" e podem ser usadas livremente pela comunidade da ufdpar?": {
        "relevantes": [
            "UFDPAR_p7_6d6df1"
        ],
        "tipo": "especifica"
    },
    "por qual canal os membros da comunidade podem reportar situações de uso inadequado de ia na ufdpar?": {
        "relevantes": [
            "UFDPAR_p11_dae0ea"
        ],
        "tipo": "especifica"
    },
    "quais as penalidades a ufdpar estipula quando ocorre descumprimento das regras?": {
        "relevantes": [
            "UFDPAR_p11_dae0ea",
            "UFDPAR_p12_8157ae"
        ],
        "tipo": "especifica"
    },
    "a ufba preve declaração de uso de IA?": {
        "relevantes": [
            "UFBA_p23_698e49",
            "UFBA_p26_24a05a",
            "UFBA_p26_423b93"
        ],
        "tipo": "especifica"
    },
    "segundo o guia da ufba, a inteligência artificial pode ser listada como autora de um trabalho?": {
        "relevantes": [
            "UFBA_p22_489efc"
        ],
        "tipo": "especifica"
    },
    "o guia da ufba tem caráter obrigatório ou apenas orientador?": {
        "relevantes": [
            "UFBA_p13_698e49"
        ],
        "tipo": "especifica"
    },
    "o que o docente da ufba deve fazer caso decida utilizar sistemas de ia na elaboração ou avaliação da turma?": {
        "relevantes": [
            "UFBA_p26_2622a2"
        ],
        "tipo": "especifica"
    },
    "onde deve estar explicitado a autorização ou não do uso de IA na UFBA?": {
        "relevantes": [
            "UFBA_p26_423b93"
        ],
        "tipo": "especifica"
    },
    "a ufmg preve declaração de uso de IA?": {
        "relevantes": [
            "UFMG_p2_1720e2",
            "UFMG_p2_a22391",
            "UFMG_p4_975146",
            "UFMG_p5_ac44c7",
            "UFMG_p4_3968b6"
        ],
        "tipo": "especifica"
    },
    "qual é a recomendação da ufmg para o registro de interações com IA em trabalhos de curso?": {
        "relevantes": [
            "UFMG_p2_a22391",
            "UFMG_p3_4bc792",
            "UFMG_p4_975146",
            "UFMG_p4_f3534c"
        ],
        "tipo": "especifica"
    },
    "o que a comissão da ufmg sugere que conste explicitamente nas ementas e planos de ensino?": {
        "relevantes": [
            "UFMG_p3_2d31f8",
            "UFMG_p3_33ce39"
        ],
        "tipo": "especifica"
    },
    "qual estrutura de governança a ufmg sugere criar para assegurar o uso responsável da tecnologia?": {
        "relevantes": [
            "UFMG_p6_16e7c9"
        ],
        "tipo": "especifica"
    },
    "quais as recomendações de uso da ia para atividades administrativas na ufmg?": {
        "relevantes": [
            "UFMG_p5_07e6dd"
        ],
        "tipo": "especifica"
    },
    "a uff recomenda o uso de ferramentas de ia para detectar plágio ou textos gerados por máquina?": {
        "relevantes": [
            "UFF_p20_8b2c4e"
        ],
        "tipo": "especifica"
    },
    "qual é a natureza jurídica do guia de ia da uff?": {
        "relevantes": [
            "UFF_p7_abf03f",
            "UFF_p7_aa5884"
        ],
        "tipo": "especifica"
    },
    "uma ferramenta de ia generativa pode ser listada como coautora de um trabalho acadêmico na uff?": {
        "relevantes": [
            "UFF_p26_bfabd8"
        ],
        "tipo": "especifica"
    },
    "em quais seções do trabalho acadêmico a uff recomenda inserir a declaração de uso de ia?": {
        "relevantes": [
            "UFF_p16_461ff6"
        ],
        "tipo": "especifica"
    },
    "quais atividades a uff permite no uso de ia?": {
        "relevantes": [
            "UFF_p18_a2eada"
        ],
        "tipo": "especifica"
    },
    "em quais tipos de atividades o uso de ferramentas de IA deve ser declarado de forma explícita na ufu?": {
        "relevantes": [
            "UFU_p10_075c52",
            "UFU_p23_671e00",
            "UFU_p17_3fff3c"
        ],
        "tipo": "especifica"
    },
    "como o estudante deve declarar o uso de IA nos seus trabalhos na ufu?": {
        "relevantes": [
            "UFU_p21_2f6543"
        ],
        "tipo": "especifica"
    },
    "como a universidade orienta os docentes a lidarem com as ferramentas de detecção de conteúdo gerado por ia na ufu?": {
        "relevantes": [
            "UFU_p21_671e00"
        ],
        "tipo": "especifica"
    },
    "é permitido aos técnicos-administrativos delegar decisões importantes exclusivamente a sistemas automatizados na ufu?": {
        "relevantes": [
            "UFU_p22_671e00"
            "UFU_p22_750c5c"
        ],
        "tipo": "especifica"
    },
    "qual é o caratér jurídico do guia de ia da ufu?": {
        "relevantes": [
            "UFU_p5_5f4392"
        ],
        "tipo": "especifica"
    },
    "qual a orientação da ufg sobre o uso de ferramentas de ia para redigir textos?": {
        "relevantes": [
            "UFG_p32_dd5046",
            "UFG_p35_0a32f5",
            "UFG_p32_d14cbc"
        ],
        "tipo": "especifica"
    },
    "qual é o órgão da ufg responsável pela disseminação de boas práticas e implantação da cultura de integridade na instituição?": {
        "relevantes": [
            "UFG_p4_444b85"
        ],
        "tipo": "especifica"
    },
    "uma ferramenta de inteligência artificial pode ser listada como autora ou coautora de um trabalho na ufg?": {
        "relevantes": [
            "UFG_p35_0a32f5"
        ],
        "tipo": "especifica"
    },
    "é permitido inserir artigos inéditos em ferramentas de ia generativa na ufg?": {
        "relevantes": [
            "UFG_p34_7e7e9d",
            "UFG_p35_0a32f5"
        ],
        "tipo": "especifica"
    },
    "qual é o caratér jurídico do guia de ia da UFG?": {
        "relevantes": [
            "UFG_p4_f20477",
            "UFG_p4_444b85"
        ],
        "tipo": "especifica"
    },
    "a ufma categoriza o que como alto risco?": {
        "relevantes": [
            "UFMA_p15_a954fd",
            "UFMA_p15_a4d186"
        ],
        "tipo": "especifica"
    },
    "uma ferramenta de inteligência artificial pode ser creditada como autora de um trabalho acadêmico na ufma?": {
        "relevantes": [
            "UFMA_p9_7c2f0d"
        ],
        "tipo": "especifica"
    },
    "onde o discente da ufma deve descrever o uso de ferramentas de ia em um trabalho acadêmico?": {
        "relevantes": [
            "UFMA_p9_7c2f0d"
        ],
        "tipo": "especifica"
    },
    "existe sanções na ufma para uso indevido de ia?": {
        "relevantes": [
            "UFMA_p9_9bdd20"
        ],
        "tipo": "especifica"
    },
    "a ufma categoriza o que como baixo risco?": {
        "relevantes": [
            "UFMA_p15_f3d7fc",
            "UFMA_p15_78a950",
            "UFMA_p15_b0d571",
            "UFMA_p15_9cc381"
        ],
        "tipo": "especifica"
    },
    "a ufop recomenda que a declaração do uso de IA seja feita em qual secção?": {
        "relevantes": [
            "UFOP_p2_5b0f82",
            "UFOP_p2_f26985",
            "UFOP_p3_78650e"
        ],
        "tipo": "especifica"
    },
    "qual é a recomendação da ufop quanto ao uso de softwares para detecção de uso de ia?": {
        "relevantes": [
            "UFOP_p2_6ca19d"
        ],
        "tipo": "especifica"
    },
    "qual é a sanção prevista na ufop?": {
        "relevantes": [
            "UFOP_p1_762e19",
            "UFOP_p2_caeee9",
            "UFOP_p2_8831fc"
        ],
        "tipo": "especifica"
    },
    "como é a declaração de uso de ia na ufop?": {
        "relevantes": [
            "UFOP_p2_5b0f82",
            "UFOP_p2_16d946",
            "UFOP_p2_a6acdb"
        ],
        "tipo": "especifica"
    },
    "quais as recomendações da ufop para o uso de ferramendas de IA no ensino?": {
        "relevantes": [
            "UFOP_p1_a077f2",
            "UFOP_p1_b16391",
            "UFOP_p1_15783c",
            "UFOP_p1_762e19",
            "UFOP_p1_39f709"
        ],
        "tipo": "especifica"
    },
    "qual é o procedimento recomendado na ufrrj caso um autor utilize IA na elaboração de um trabalho acadêmico?": {
        "relevantes": [
            "UFRRJ_p11_7f9fe3"
        ],
        "tipo": "especifica"
    },
    "uma ferramenta de Inteligência Artificial pode ser considerada autora de um trabalho acadêmico na ufrrj?": {
        "relevantes": [
            "UFRRJ_p11_7f9fe3",
            "UFRRJ_p11_105047"
        ],
        "tipo": "especifica"
    },
    "onde os docentes da ufrrj devem delimitar as formas permitidas de uso da ia em suas disciplinas?": {
        "relevantes": [
            "UFRRJ_p11_7f9fe3"
        ],
        "tipo": "especifica"
    },
    "de acordo com o guia da ufrrj, o usuário pode adotar decisões automatizadas da ia?": {
        "relevantes": [
            "UFRRJ_p11_105047"
        ],
        "tipo": "especifica"
    },
    "qual a natureza juridica do documento da ufrrj?": {
        "relevantes": [
            "UFRRJ_p5_039080",
            "UFRRJ_p6_999971"
        ],
        "tipo": "especifica"
    },
    "a unifal preve declaração de uso de IA?": {
        "relevantes": [
            "UNIFAL MG_p2_e3dbc7",
            "UNIFAL MG_p2_b387d2",
            "UNIFAL MG_p2_8f57a0"
        ],
        "tipo": "especifica"
    },
    "é permitido na unifal o uso de ia para a geração de dados científicos ou resultados experimentais na pesquisa e pós-graduação?": {
        "relevantes": [
            "UNIFAL MG_p2_28bb15"
        ],
        "tipo": "especifica"
    },
    "onde devem ser explicitados os usos permitidos e vedados de ia no âmbito do ensino?": {
        "relevantes": [
            "UNIFAL MG_p2_e3dbc7"
        ],
        "tipo": "especifica"
    },
    "o que a unifal fala sobre o uso de ia na extensão?": {
        "relevantes": [
            "UNIFAL MG_p3_77056d"
        ],
        "tipo": "especifica"
    },
    "o que a unifal fala sobre o uso de ia de forma geral?": {
        "relevantes": [
            "UNIFAL MG_p2_b387d2"
        ],
        "tipo": "especifica"
    },
    "a unifesp preve declaração de uso de ia?": {
        "relevantes": [
            "UNIFESP_p2_3e30ad"
        ],
        "tipo": "especifica"
    },
    "é permitido que decisões acadêmicas sejam tomadas de forma automática por ferramentas de iag na unifesp?": {
        "relevantes": [
            "UNIFESP_p2_e33d7c"
        ],
        "tipo": "especifica"
    },
    "para qual instância da unifesp devem ser encaminhados eventuais indícios de uso inadequado de iag?": {
        "relevantes": [
            "UNIFESP_p2_e33d7c"
        ],
        "tipo": "especifica"
    },
    "onde os discentes e pesquisadores podem encontrar o modelo oficial da declaração de uso de ia da unifesp?": {
        "relevantes": [
            "UNIFESP_p1_6306a2",
            "UNIFESP_p1_1292ca"
        ],
        "tipo": "especifica"
    },
    "a unifesp permite listar ferramentas de ia como coautoras?": {
        "relevantes": [
            "UNIFESP_p2_3e30ad"
        ],
        "tipo": "especifica"
    },
    "a unir preve declaração de uso de ia?": {
        "relevantes": [
            "UNIR_p1_e5ac56"
        ],
        "tipo": "especifica"
    },
    "é permitido listar uma ferramenta de Inteligência Artificial como coautora de um trabalho acadêmico na unir?": {
        "relevantes": [
            "UNIR_p1_159395"
        ],
        "tipo": "especifica"
    },
    "na unir, qual o prazo mínimo para o autor apresentar manifestação escrita após ser notificado por suspeita de uso indevido de ia?": {
        "relevantes": [
            "UNIR_p2_eec43d"
        ],
        "tipo": "especifica"
    },
    "a IA pode ser utilizada para a produção de dados experimentais unir?": {
        "relevantes": [
            "UNIR_p1_159395",
            "UNIR_p1_ffac3c"
        ],
        "tipo": "especifica"
    },
    "quais os usos vedados da ia na unir?": {
        "relevantes": [
            "UNIR_p1_159395",
            "UNIR_p1_ffac3c"
        ],
        "tipo": "especifica"
    },
    "no que se refere à verificação de similaridade, qual a diferença entre a abordagem da ufpb e a da ufc?": {
        "relevantes": [
            "UFC_p2_71635f",
            "UFC_p1_d0343c",
            "UFC_p3_bcf4ba",
            "UFPB_p3_6915da",
            "UFPB_p4_295251",
            "UFPB_p4_295251"
        ],
        "tipo": "comparativa"
    },
    "compare a politica de uso de ia da ufu com a da unifal": {
        "relevantes": [
            "UFU_p5_d0a010",
            "UFU_p5_5f4392",
            "UNIFAL MG_p1_d5a719"
        ],
        "tipo": "comparativa"
    },
    "quanto à natureza jurídica dos documentos, como se diferenciam as orientações da ufrrj em relação às normas da ufma?": {
        "relevantes": [
            "UFMA_p1_b5dea8",
            "UFRRJ_p5_039080",
            "UFRRJ_p6_999971",
            "UFRRJ_p6_f631c0",
            "UFMA_p1_c94f19"
        ],
        "tipo": "comparativa"
    },
    "onde deve ser inserida a declaração de uso de ia segundo as diretrizes da unir e da ufma?": {
        "relevantes": [
            "UNIR_p1_e5ac56",
            "UFMA_p9_7c2f0d"
        ],
        "tipo": "comparativa"
    },
    "como se estruturam os modelos de governança de ia na ufmg, ufms e ufdpar?": {
        "relevantes": [
            "UFMG_p6_037bad",
            "UFMG_p6_16e7c9",
            "UFMS_p3_5eb211",
            "UFMS_p1_d1a5d9",
            "UFMS_p3_d2070f",
            "UFDPAR_p9_a78b46",
            "UFDPAR_p10_8157ae",
            "UFDPAR_p9_410dc5"
        ],
        "tipo": "comparativa"
    },
    "qual é o posicionamento da uff e da ufop sobre o uso de softwares de detecção de escrita por ia?": {
        "relevantes": [
            "UFOP_p2_6ca19d",
            "UFF_p20_8b2c4e"
        ],
        "tipo": "comparativa"
    },
    "quanto à natureza e ao objetivo principal, qual a semelhança entre os documentos da ufba e da ufg?": {
        "relevantes": [
            "UFBA_p13_14e611",
            "UFG_p4_f20477"
        ],
        "tipo": "comparativa"
    },
    "o que a unifesp e a ufma falam sobre a autoria?": {
        "relevantes": [
            "UNIFESP_p2_3e30ad",
            "UFMA_p9_7c2f0d"
        ],
        "tipo": "comparativa"
    },
    "qual a semelhança básica entre a natureza dos documentos da ufba e da uff?": {
        "relevantes": [
            "UFBA_p13_14e611",
            "UFF_p7_abf03f"
        ],
        "tipo": "comparativa"
    },
    "o que a ufg e a ufms falam sobre a autoria?": {
        "relevantes": [
            "UFG_p35_0a32f5",
            "UFMS_p6_0b03ec"
        ],
        "tipo": "comparativa"
    }
}

print(f"Gabarito com {len(GABARITO)} perguntas anotadas:")
tipos = {}
for q, v in GABARITO.items():
    t = v["tipo"]
    tipos[t] = tipos.get(t, 0) + 1
for t, n in sorted(tipos.items()):
    print(f"  {t:<12}: {n} perguntas")

In [ ]:
import numpy as np
import hashlib
from collections import defaultdict

def chunk_id(chunk) -> str:
    uni = chunk.metadata["universidade"]
    pag = chunk.metadata["pagina"]
    h   = hashlib.md5(chunk.page_content[:100].encode()).hexdigest()[:6]
    return f"{uni}_p{pag}_{h}"

def precision_at_k(ids_rec, ids_rel, k):
    hits = sum(1 for cid in ids_rec[:k] if cid in ids_rel)
    return hits / k if k > 0 else 0.0

def recall_at_k(ids_rec, ids_rel, k):
    if not ids_rel: return 0.0
    hits = sum(1 for cid in ids_rec[:k] if cid in ids_rel)
    return hits / len(ids_rel)

def mrr(ids_rec, ids_rel):
    for i, cid in enumerate(ids_rec):
        if cid in ids_rel:
            return 1 / (i + 1)
    return 0.0

# ── Variantes do pipeline ─────────────────────────────────────────────────────

def retrieval_bm25(query, top_k):
    return bm25_retriever.invoke(query)[:top_k]

def retrieval_chroma_global(query, top_k):
    return chroma_store.similarity_search(query, k=top_k)

def retrieval_chroma_filtrado(query, top_k, unis):
    """Fallback para global se não há universidade na query."""
    if not unis:
        return chroma_store.similarity_search(query, k=top_k)
    resultados = []
    for uni in unis:
        resultados += chroma_filtrado(query, uni, top_k)
    # deduplica mantendo ordem
    vistos, dedup = set(), []
    for c in resultados:
        cid = chunk_id(c)
        if cid not in vistos:
            vistos.add(cid)
            dedup.append(c)
    return dedup[:top_k]

def retrieval_hibrido_sem_reranker(query, top_k):
    candidatos = retrieval_hibrido(query)
    return candidatos[:top_k]

VARIANTES = {
    "bm25":               lambda query, top_k, unis: retrieval_bm25(query, top_k),
    "chroma_global":      lambda query, top_k, unis: retrieval_chroma_global(query, top_k),
    "chroma_filtrado":    lambda query, top_k, unis: retrieval_chroma_filtrado(query, top_k, unis),
    "hibrido":            lambda query, top_k, unis: retrieval_hibrido_sem_reranker(query, top_k),
    "bm25+reranker":      lambda query, top_k, unis: rerankar(query, bm25_retriever.invoke(query), top_k),
    "chroma+reranker":    lambda query, top_k, unis: rerankar(query, chroma_store.similarity_search(query, k=TOP_K_EACH), top_k),
    "hibrido+reranker":   lambda query, top_k, unis: rerankar(query, retrieval_hibrido(query), top_k),
}

# ── Roda o ablation ───────────────────────────────────────────────────────────

def rodar_ablation(gabarito: dict) -> dict:
    """
    Para cada variante e cada pergunta do gabarito, calcula P@K, R@K e MRR.
    Retorna dict: variante -> lista de resultados por pergunta.
    """
    # resultados[variante] = [{"tipo": ..., "precision": ..., "recall": ..., "mrr": ...}, ...]
    resultados = {v: [] for v in VARIANTES}

    n = len(gabarito)
    for i, (pergunta, anotacao) in enumerate(gabarito.items(), 1):
        ids_relevantes = set(anotacao["relevantes"])
        tipo           = anotacao["tipo"]
        unis           = extrair_universidades_da_query(pergunta)
        _, top_k       = classificar_query(unis)

        print(f"  [{i:>2}/{n}] {pergunta[:55]}")

        for nome, fn in VARIANTES.items():
            chunks_rec = fn(pergunta, top_k, unis)
            ids_rec    = [chunk_id(c) for c in chunks_rec]
            resultados[nome].append({
                "tipo":      tipo,
                "top_k":     top_k,
                "precision": precision_at_k(ids_rec, ids_relevantes, top_k),
                "recall":    recall_at_k(ids_rec, ids_relevantes, top_k),
                "mrr":       mrr(ids_rec, ids_relevantes),
            })

    return resultados


def imprimir_ablation(resultados: dict):
    SEP  = "=" * 75
    SEP2 = "-" * 75

    # ── Tabela 1: visão geral por variante ───────────────────────────────────
    print(f"\n{SEP}")
    print("  ABLATION STUDY — VISÃO GERAL (todas as perguntas)")
    print(SEP)
    print(f"  {'Variante':<22} {'N':>4}  {'P@K':>6}  {'R@K':>6}  {'MRR':>6}")
    print(f"  {'-'*22} {'-'*4}  {'-'*6}  {'-'*6}  {'-'*6}")

    for nome, rows in VARIANTES.items():
        data = resultados[nome]
        p = np.mean([r["precision"] for r in data])
        r = np.mean([r["recall"]    for r in data])
        m = np.mean([r["mrr"]       for r in data])
        destaque = " <-" if nome == "hibrido+reranker" else ""
        print(f"  {nome:<22} {len(data):>4}  {p:>6.3f}  {r:>6.3f}  {m:>6.3f}{destaque}")

    print(SEP)

    # ── Tabela 2: por tipo de pergunta ───────────────────────────────────────
    tipos = sorted(set(r["tipo"] for rows in resultados.values() for r in rows))

    print(f"\n{SEP}")
    print("  ABLATION STUDY — POR TIPO DE PERGUNTA")

    for tipo in tipos:
        print(f"\n{SEP2}")
        print(f"  Tipo: {tipo.upper()}")
        print(f"  {'Variante':<22} {'N':>4}  {'P@K':>6}  {'R@K':>6}  {'MRR':>6}")
        print(f"  {'-'*22} {'-'*4}  {'-'*6}  {'-'*6}  {'-'*6}")

        for nome in VARIANTES:
            data = [r for r in resultados[nome] if r["tipo"] == tipo]
            if not data:
                continue
            p = np.mean([r["precision"] for r in data])
            r = np.mean([r["recall"]    for r in data])
            m = np.mean([r["mrr"]       for r in data])
            # avisa se chroma_filtrado usou fallback (tipo geral)
            nota = " (fallback global)" if nome == "chroma_filtrado" and tipo == "geral" else ""
            print(f"  {nome:<22} {len(data):>4}  {p:>6.3f}  {r:>6.3f}  {m:>6.3f}{nota}")


# ── Executa ───────────────────────────────────────────────────────────────────
print("Rodando ablation study...\n")
resultados_ablation = rodar_ablation(GABARITO)
imprimir_ablation(resultados_ablation)

Rodando ablation study...

  [ 1/90] a ufpb permite o uso de ia para formatar textos?
  [ 2/90] qual é a comissão responsável por analisar denúncias de
  [ 3/90] qual o percentual de similaridade em um trabalho que é 
  [ 4/90] quais são as possíveis sanções aplicáveis em caso de pr
  [ 5/90] quais os usos de ia vedados na ufpb?
  [ 6/90] qual a ferramenta de detecção de similaridade recomenda
  [ 7/90] quais são os documentos obrigatórios que devem ser anex
  [ 8/90] pode utilizar ia para redigir conclusões na ufc?
  [ 9/90] na ufc, o que ocorre se o docente não explicitar no pla
  [10/90] quais os usos de ia vedados na ufc?
  [11/90] a declaração de uso da ia é obrigatoria na ufms?
  [12/90] ferramentas de inteligência artificial podem ser listad
  [13/90] quais são os órgãos responsáveis pela governança instit
  [14/90] como o estudante deve agir caso não haja instrução form
  [15/90] como a ia pode ser usada nas atividades administrativas
  [16/90] qual o nome da comissão responsáv

In [ ]:
import numpy as np
import hashlib

def chunk_id(chunk) -> str:
    """ID estável e único por chunk: universidade + página + hash do conteúdo."""
    uni = chunk.metadata["universidade"]
    pag = chunk.metadata["pagina"]
    h   = hashlib.md5(chunk.page_content[:100].encode()).hexdigest()[:6]
    return f"{uni}_p{pag}_{h}"

def precision_at_k(ids_recuperados: list, ids_relevantes: set, k: int) -> float:
    """Dos K chunks recuperados, quantos eram relevantes?"""
    top_k = ids_recuperados[:k]
    hits  = sum(1 for cid in top_k if cid in ids_relevantes)
    return hits / k if k > 0 else 0.0

def recall_at_k(ids_recuperados: list, ids_relevantes: set, k: int) -> float:
    """Dos chunks relevantes existentes, quantos foram recuperados no top-K?"""
    if not ids_relevantes:
        return 0.0
    top_k = ids_recuperados[:k]
    hits  = sum(1 for cid in top_k if cid in ids_relevantes)
    return hits / len(ids_relevantes)

def mrr(ids_recuperados: list, ids_relevantes: set) -> float:
    """Em que posição aparece o primeiro chunk relevante? (1/posição)"""
    for i, cid in enumerate(ids_recuperados):
        if cid in ids_relevantes:
            return 1 / (i + 1)
    return 0.0

def avaliar_pipeline(gabarito: dict) -> list:
    """
    Roda o pipeline para cada pergunta e calcula as métricas.
    Usa IDs estáveis (universidade_pPagina_hash) para comparar com o gabarito.
    """
    resultados = []
    avisos     = []

    print("Rodando pipeline para cada pergunta do gabarito...\n")

    for pergunta, anotacao in gabarito.items():
        ids_relevantes = set(anotacao["relevantes"])
        tipo           = anotacao["tipo"]

        unis = extrair_universidades_da_query(pergunta)
        _, top_k = classificar_query(unis)
        candidatos    = retrieval_hibrido(pergunta)
        chunks_finais = rerankar(pergunta, candidatos, top_k)

        ids_rec        = [chunk_id(c) for c in chunks_finais]
        ids_candidatos = set(chunk_id(c) for c in candidatos)
        ids_ausentes   = ids_relevantes - ids_candidatos
        if ids_ausentes:
            avisos.append((pergunta, ids_ausentes))

        p_k  = precision_at_k(ids_rec, ids_relevantes, top_k)
        r_k  = recall_at_k(ids_rec, ids_relevantes, top_k)
        mrr_ = mrr(ids_rec, ids_relevantes)

        resultados.append({
            "pergunta":  pergunta,
            "tipo":      tipo,
            "top_k":     top_k,
            "precision": p_k,
            "recall":    r_k,
            "mrr":       mrr_,
        })

        status = "OK  " if mrr_ > 0 else "MISS"
        print(f"  [{status}] [{tipo:<11}] P@{top_k}={p_k:.2f} R@{top_k}={r_k:.2f} MRR={mrr_:.2f} | {pergunta[:50]}")

    if avisos:
        print("\nChunks do gabarito ausentes nos candidatos (pipeline não os trouxe):")
        for p, ids in avisos:
            print(f"   {p[:50]} -> {ids}")
        print("   -> Considere aumentar TOP_K_EACH ou revisar esses IDs no gabarito.")

    return resultados


def imprimir_relatorio(resultados: list):
    """Imprime relatório agregado por tipo e geral."""
    SEP = "=" * 65

    tipos = {}
    for r in resultados:
        tipos.setdefault(r["tipo"], []).append(r)

    print(f"\n{SEP}")
    print(f"  RELATORIO DE AVALIACAO DO RETRIEVER")
    print(SEP)
    print(f"  {'Tipo':<12} {'N':>4}  {'P@K':>6}  {'R@K':>6}  {'MRR':>6}")
    print(f"  {'-'*12} {'-'*4}  {'-'*6}  {'-'*6}  {'-'*6}")

    all_p, all_r, all_m = [], [], []

    for tipo, grupo in sorted(tipos.items()):
        p = np.mean([r["precision"] for r in grupo])
        r = np.mean([r["recall"]    for r in grupo])
        m = np.mean([r["mrr"]       for r in grupo])
        all_p += [r["precision"] for r in grupo]
        all_r += [r["recall"]    for r in grupo]
        all_m += [r["mrr"]       for r in grupo]
        print(f"  {tipo:<12} {len(grupo):>4}  {p:>6.3f}  {r:>6.3f}  {m:>6.3f}")

    print(f"  {'─'*12} {'─'*4}  {'─'*6}  {'─'*6}  {'─'*6}")
    print(f"  {'GERAL':<12} {len(resultados):>4}  {np.mean(all_p):>6.3f}  {np.mean(all_r):>6.3f}  {np.mean(all_m):>6.3f}")
    print(SEP)

    piores   = sorted(resultados, key=lambda x: x["mrr"])[:3]
    melhores = sorted(resultados, key=lambda x: x["mrr"], reverse=True)[:3]

    print("\n  Piores casos (MRR mais baixo):")
    for r in piores:
        print(f"    MRR={r['mrr']:.2f} [{r['tipo']}] {r['pergunta'][:55]}")

    print("\n  Melhores casos (MRR mais alto):")
    for r in melhores:
        print(f"    MRR={r['mrr']:.2f} [{r['tipo']}] {r['pergunta'][:55]}")



# ── Roda a avaliação ──────────────────────────────────────────────────────
resultados_avaliacao = avaliar_pipeline(GABARITO)
imprimir_relatorio(resultados_avaliacao)


Rodando pipeline para cada pergunta do gabarito...

  [OK  ] [especifica ] P@5=0.20 R@5=1.00 MRR=1.00 | a ufpb permite o uso de ia para formatar textos?
  [OK  ] [especifica ] P@5=0.20 R@5=0.33 MRR=0.50 | qual é a comissão responsável por analisar denúnci
  [OK  ] [especifica ] P@5=0.40 R@5=1.00 MRR=1.00 | qual o percentual de similaridade em um trabalho q
  [OK  ] [especifica ] P@5=0.40 R@5=1.00 MRR=0.50 | quais são as possíveis sanções aplicáveis em caso 
  [OK  ] [especifica ] P@5=0.20 R@5=0.50 MRR=0.33 | quais os usos de ia vedados na ufpb?
  [OK  ] [especifica ] P@5=0.20 R@5=1.00 MRR=0.25 | qual a ferramenta de detecção de similaridade reco
  [OK  ] [especifica ] P@5=0.40 R@5=1.00 MRR=1.00 | quais são os documentos obrigatórios que devem ser
  [MISS] [especifica ] P@5=0.00 R@5=0.00 MRR=0.00 | pode utilizar ia para redigir conclusões na ufc?
  [OK  ] [especifica ] P@5=0.20 R@5=1.00 MRR=0.25 | na ufc, o que ocorre se o docente não explicitar n
  [MISS] [especifica ] P@5=0.00 R@5=0.0

In [ ]:
# ── 11d. Salva resultados em CSV ─────────────────────────────────────────────

import csv

caminho_csv = "/content/avaliacao_retriever.csv"

with open(caminho_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "pergunta", "tipo", "top_k", "precision", "recall", "mrr"
    ])
    writer.writeheader()
    for r in resultados_avaliacao:
        writer.writerow({
            "pergunta": r["pergunta"],
            "tipo":     r["tipo"],
            "top_k":    r["top_k"],
            "precision": round(r["precision"], 4),
            "recall":    round(r["recall"],    4),
            "mrr":       round(r["mrr"],       4),
        })

print(f"Resultados salvos em {caminho_csv}")

# Download direto no Colab
from google.colab import files
files.download(caminho_csv)

#10. Avaliação Comparativa

In [ ]:
import re
import string
import time
import random
import numpy as np
import hashlib
from collections import Counter
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

MODELO_TESTE = "llama-3.1-8b-instant"
N_AMOSTRAS   = 90
SEED         = 42
SLEEP_S      = 1.5   # segundos entre cada par de chamadas


llm_original = llm
llm = ChatGroq(model=MODELO_TESTE, temperature=0.1, groq_api_key=GROQ_API_KEY)
print(f"Modelo : {MODELO_TESTE}")
print(f"Sleep  : {SLEEP_S}s entre perguntas\n")


random.seed(SEED)
perguntas = random.sample(list(GABARITO.keys()), N_AMOSTRAS)


def chunk_id(chunk) -> str:
    uni = chunk.metadata["universidade"]
    pag = chunk.metadata["pagina"]
    h   = hashlib.md5(chunk.page_content[:100].encode()).hexdigest()[:6]
    return f"{uni}_p{pag}_{h}"


def normalizar(texto: str) -> str:
    texto = texto.lower()
    texto = texto.translate(str.maketrans("", "", string.punctuation))
    artigos = {"o","a","os","as","um","uma","uns","umas","de","do","da","dos","das","e","em","no","na","nos","nas"}
    return " ".join(t for t in texto.split() if t not in artigos)

def squad_f1(referencia: str, hipotese: str) -> dict:
    ref_tokens = normalizar(referencia).split()
    hip_tokens = normalizar(hipotese).split()

    if not ref_tokens or not hip_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    comum = sum((Counter(ref_tokens) & Counter(hip_tokens)).values())

    if comum == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    precision = comum / len(hip_tokens)
    recall    = comum / len(ref_tokens)
    f1        = 2 * precision * recall / (precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}


def responder_sem_rag(pergunta: str) -> str:
    return llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=pergunta)
    ]).content


print(f"Avaliando {N_AMOSTRAS} perguntas...\n")
print(f"  {'#':>3}  {'P-RAG':>6}  {'R-RAG':>6}  {'F1-RAG':>6}  {'P-Base':>6}  {'R-Base':>6}  {'F1-Base':>6}  Pergunta")
print(f"  {'─'*3}  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*40}")

scores_rag, scores_base = [], []

for i, pergunta in enumerate(perguntas, 1):
    # Referência
    ids_relevantes = set(GABARITO[pergunta]["relevantes"])
    textos = [c.page_content for c in chunks if chunk_id(c) in ids_relevantes]
    referencia = " ".join(textos) if textos else ""

    # Respostas
    r_rag  = responder(pergunta, verbose=False)["resposta"]
    r_base = responder_sem_rag(pergunta)

    s_rag  = squad_f1(referencia, r_rag)
    s_base = squad_f1(referencia, r_base)

    scores_rag.append(s_rag)
    scores_base.append(s_base)

    print(
        f"  {i:>3}  "
        f"{s_rag['precision']:>6.3f}  {s_rag['recall']:>6.3f}  {s_rag['f1']:>6.3f}  "
        f"{s_base['precision']:>6.3f}  {s_base['recall']:>6.3f}  {s_base['f1']:>6.3f}  "
        f"{pergunta[:45]}"
    )

    time.sleep(SLEEP_S)

SEP = "=" * 72

print(f"\n{SEP}")
print(f"  RESULTADO FINAL — SQuAD-style (arxiv 2411.13691)")
print(f"  Modelo: {MODELO_TESTE}  |  N = {N_AMOSTRAS} perguntas  |  seed = {SEED}")
print(f"  Referência = chunks do GABARITO")
print(SEP)
print(f"  {'Métrica':<12} {'COM RAG':>9}  {'SEM RAG':>9}  {'Δ (RAG-Base)':>13}")
print(f"  {'─'*12} {'─'*9}  {'─'*9}  {'─'*13}")

for metrica in ["precision", "recall", "f1"]:
    m_rag  = np.mean([s[metrica] for s in scores_rag])
    m_base = np.mean([s[metrica] for s in scores_base])
    diff   = m_rag - m_base
    sinal  = "+" if diff >= 0 else ""
    print(f"  {metrica.capitalize():<12} {m_rag:>9.3f}  {m_base:>9.3f}  {sinal}{diff:>12.3f}")

print(SEP)

llm = llm_original
print(f"LLM restaurado para: {LLM_MODEL}")